# FAQ RAG — pergunta → resposta com LLM

Este notebook **reutiliza o índice** construído em `uc15_faq_notebook5_arthur.ipynb` (mesmo `persist_directory` e modelo de embeddings).

**Fluxo:** pergunta do usuário → recuperação em 2 estágios (denso + *rerank*) → o LLM gera uma resposta **somente com base nos trechos** recuperados.

**LLM:** defina `OPENAI_API_KEY` para usar `gpt-4o-mini`, ou use **Ollama** local (variáveis `OLLAMA_BASE_URL` e `OLLAMA_MODEL` opcionais).

In [1]:
# Descomente se faltar dependência:
# %pip install -q langchain-core langchain-community langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers pypdf torch
# %pip install -q langchain-openai
# %pip install -q langchain-ollama

## Configuração e carregamento do vector store

In [2]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Sequence

import torch
from sentence_transformers import CrossEncoder

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data").is_dir() and (p / "notebooks").is_dir():
            return p
    return here


PROJECT_ROOT = find_project_root()


@dataclass
class FaqRagConfig:
    persist_directory: str = "./chroma_langchain_db_arthur_faq_2stage"
    collection_name: str = "faq_sindilojas_2stage"
    embedding_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    reranker_model_name: str = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
    initial_k: int = 20
    final_k: int = 5
    device: Optional[str] = None
    normalize_embeddings: bool = True
    reranker_batch_size: int = 8


def get_device(explicit: Optional[str] = None) -> str:
    if explicit:
        return explicit
    return "cuda" if torch.cuda.is_available() else "cpu"


config = FaqRagConfig()
device = get_device(config.device)

embeddings = HuggingFaceEmbeddings(
    model_name=config.embedding_model_name,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
)

vector_store = Chroma(
    collection_name=config.collection_name,
    embedding_function=embeddings,
    persist_directory=config.persist_directory,
)

n_docs = vector_store._collection.count()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Persist dir:", config.persist_directory)
print("Chunks no índice:", n_docs)
if n_docs == 0:
    print(
        "Índice vazio: execute primeiro o notebook 5 (ingestão) ou descomente a célula opcional abaixo."
    )

OSError: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

### (Opcional) Ingerir PDF se o índice estiver vazio

Só rode esta célula se ainda não tiver executado o notebook 5.

In [ ]:
# import hashlib
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
#
# pdf_path = PROJECT_ROOT / "data" / "marcelo" / "sindilojas_2025_2026.pdf"
# loader = PyPDFLoader(str(pdf_path))
# docs = loader.load()
# splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100, add_start_index=True)
# splits = splitter.split_documents(docs)
#
# def deterministic_chunk_id(source: str, page: int, chunk_text: str) -> str:
#     payload = f"{source}|{page}|{chunk_text.strip().lower()}"
#     return hashlib.sha256(payload.encode("utf-8")).hexdigest()
#
# ids = [
#     deterministic_chunk_id(str(c.metadata.get("source", "")), int(c.metadata.get("page", 0) or 0), c.page_content)
#     for c in splits
# ]
# vector_store.add_documents(documents=splits, ids=ids)
# print("Indexados:", len(ids))

## Retriever em 2 estágios (igual ao notebook 5)

In [ ]:
class TwoStageRetriever:
    """Stage 1: Chroma (bi-encoder). Stage 2: cross-encoder rerank."""

    def __init__(
        self,
        vectorstore: Chroma,
        reranker_model_name: str,
        initial_k: int = 20,
        final_k: int = 5,
        device: Optional[str] = None,
        reranker_batch_size: int = 8,
    ) -> None:
        self.vectorstore = vectorstore
        self.initial_k = initial_k
        self.final_k = final_k
        self.device = get_device(device)
        self.reranker_batch_size = reranker_batch_size
        self.cross_encoder = CrossEncoder(reranker_model_name, device=self.device)

    def _dense_retrieve(self, query: str) -> List[Document]:
        return self.vectorstore.similarity_search(query, k=self.initial_k)

    def _rerank(self, query: str, docs: Sequence[Document]) -> List[Document]:
        if not docs:
            return []
        pairs = [(query, d.page_content) for d in docs]
        scores = self.cross_encoder.predict(
            pairs,
            batch_size=self.reranker_batch_size,
            show_progress_bar=False,
        )
        scored: List[Document] = []
        for doc, score in zip(docs, scores):
            meta = dict(doc.metadata) if doc.metadata else {}
            meta["reranker_score"] = float(score)
            scored.append(Document(page_content=doc.page_content, metadata=meta))
        scored.sort(key=lambda d: d.metadata["reranker_score"], reverse=True)
        return scored[: self.final_k]

    def invoke(self, query: str) -> List[Document]:
        return self._rerank(query, self._dense_retrieve(query))

    def as_runnable(self) -> RunnableLambda:
        return RunnableLambda(self.invoke)


retriever = TwoStageRetriever(
    vectorstore=vector_store,
    reranker_model_name=config.reranker_model_name,
    initial_k=config.initial_k,
    final_k=config.final_k,
    device=config.device,
    reranker_batch_size=config.reranker_batch_size,
)
print("Retriever pronto.")

## LLM + cadeia RAG (LCEL)

In [ ]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


def build_llm():
    """OpenAI se houver API key; senão Ollama (local)."""
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(model="gpt-4o-mini", temperature=0)
    try:
        from langchain_ollama import ChatOllama
    except ImportError:
        from langchain_community.chat_models import ChatOllama

    base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    model = os.getenv("OLLAMA_MODEL", "llama3.2")
    return ChatOllama(model=model, base_url=base_url, temperature=0)


llm = build_llm()
if os.getenv("OPENAI_API_KEY"):
    print("LLM: OpenAI (gpt-4o-mini)")
else:
    print(
        "LLM: Ollama",
        os.getenv("OLLAMA_MODEL", "llama3.2"),
        "em",
        os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
    )


def format_context(docs: Sequence[Document]) -> str:
    parts: List[str] = []
    for i, d in enumerate(docs, 1):
        page = d.metadata.get("page", "?")
        score = d.metadata.get("reranker_score")
        head = f"--- Trecho {i} (pág. {page})"
        if score is not None:
            head += f", score_rerank={score:.3f}"
        head += " ---"
        parts.append(head + "\n" + d.page_content)
    return "\n\n".join(parts)


SYSTEM = """Você é um assistente de FAQ sobre a convenção coletiva (documento Sindilojas).
Responda em português do Brasil usando **apenas** o contexto abaixo.
Se o contexto não contiver informação suficiente, diga claramente que não é possível responder com os trechos fornecidos.
Seja objetivo e cite números/cláusulas quando aparecerem no contexto."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM),
        ("human", "Contexto:\n{context}\n\nPergunta: {question}"),
    ]
)

rag_chain = (
    RunnablePassthrough.assign(
        context=itemgetter("question") | retriever | format_context
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("Cadeia RAG pronta.")

## Pergunta → resposta

In [ ]:
def answer_question(question: str) -> str:
    """Recebe a pergunta do usuário e devolve a resposta do LLM."""
    return rag_chain.invoke({"question": question.strip()})


pergunta = (
    "Quais obrigações a empresa deve cumprir em caso de parcelamento da rescisão?"
)
resposta = answer_question(pergunta)
print(resposta)